# Modelo 2 · Clasificación: supervivencia en el Titanic

**Objetivo:** predecir una **categoría** (sobrevivió: sí/no) a partir de los datos de cada pasajero.

**Dataset:** *Titanic* (el clásico de Kaggle). Lo cargamos desde `seaborn`, así que no se necesita cuenta de Kaggle.

| Columna | Descripción |
|---|---|
| `pclass` | Clase del pasaje (1, 2, 3) |
| `sex` | Sexo |
| `age` | Edad (tiene valores faltantes) |
| `sibsp` | Hermanos o cónyuges a bordo |
| `parch` | Padres o hijos a bordo |
| `fare` | Tarifa pagada |
| `embarked` | Puerto de embarque (C, Q, S) |
| **`survived`** | **Objetivo:** 1 = sobrevivió, 0 = no |

**Novedad respecto a regresión:** hay que **limpiar datos** (faltantes y variables categóricas) y usar métricas de clasificación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 1. Cargar los datos

In [ ]:
df = sns.load_dataset("titanic")
columnas = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]
df = df[columnas]
print(df.shape)
df.head()

## 2. Exploración (EDA)

In [ ]:
print("Valores faltantes por columna:")
df.isna().sum()

In [ ]:
print("Proporción de cada clase:")
df["survived"].value_counts(normalize=True).round(3)

> Si un modelo siempre dijera "no sobrevivió", tendría ~62% de accuracy. Ese es el **baseline** que hay que superar.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=df, x="sex", y="survived", ax=axes[0])
sns.barplot(data=df, x="pclass", y="survived", ax=axes[1])
sns.histplot(data=df, x="age", hue="survived", multiple="stack", bins=30, ax=axes[2])
axes[0].set_title("Supervivencia por sexo")
axes[1].set_title("Supervivencia por clase")
axes[2].set_title("Edad según supervivencia")
plt.tight_layout()
plt.show()

## 3. Separar en entrenamiento y prueba

Usamos `stratify=y` para que train y test tengan la misma proporción de sobrevivientes.

In [ ]:
X = df.drop(columns="survived")
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 4. Preprocesamiento

Los modelos solo entienden números y no aceptan valores faltantes:

- **Numéricas:** rellenar faltantes con la mediana y estandarizar.
- **Categóricas:** rellenar con el valor más frecuente y aplicar *one-hot encoding*.

Lo ponemos dentro de un `Pipeline` para que el preprocesamiento se **aprenda solo con train** (evita *data leakage*).

In [ ]:
numericas = ["age", "sibsp", "parch", "fare"]
categoricas = ["pclass", "sex", "embarked"]

preprocesamiento = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categoricas),
])

## 5. Entrenar modelos

In [ ]:
modelos = {
    "Regresión logística": Pipeline([
        ("prep", preprocesamiento),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocesamiento),
        ("modelo", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE)),
    ]),
}

predicciones = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    predicciones[nombre] = modelo.predict(X_test)
    print(f"{nombre}: accuracy = {accuracy_score(y_test, predicciones[nombre]):.3f}")

## 6. Evaluar

La accuracy sola no basta. Miramos también:

- **Precision:** de los que el modelo dijo "sobrevive", ¿cuántos sobrevivieron realmente?
- **Recall:** de los que sobrevivieron, ¿cuántos detectó el modelo?
- **F1:** promedio armónico entre precision y recall.
- **Matriz de confusión:** dónde se equivoca el modelo.

In [ ]:
for nombre, pred in predicciones.items():
    print(f"=== {nombre} ===")
    print(classification_report(y_test, pred, target_names=["No sobrevivió", "Sobrevivió"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nombre, pred) in zip(axes, predicciones.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["No", "Sí"], cmap="Blues", ax=ax, colorbar=False
    )
    ax.set_title(nombre)
plt.tight_layout()
plt.show()

## 7. Interpretar

In [ ]:
logistica = modelos["Regresión logística"]
nombres = logistica.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(logistica.named_steps["modelo"].coef_[0], index=nombres).sort_values()
coefs.plot.barh(title="Coeficientes de la regresión logística")
plt.show()

## Conclusiones

- El **sexo** y la **clase** son las variables que más influyen ("mujeres y niños primero" se ve en los datos).
- Ambos modelos superan claramente el baseline de ~62%.
- El preprocesamiento (faltantes + categóricas) es parte del modelo, no un paso aparte.

## Ejercicios

1. Entrena un modelo **sin** `stratify=y`. ¿Cambia la proporción de clases en test?
2. Agrega una variable `family_size = sibsp + parch + 1`. ¿Mejora el modelo?
3. Usa `predict_proba` y cambia el umbral de 0.5 a 0.3. ¿Qué pasa con el recall y la precision de la clase "Sobrevivió"?
4. Prueba un `DecisionTreeClassifier` con `max_depth=3` y dibújalo con `sklearn.tree.plot_tree`. ¿Qué reglas aprendió?